# Train LightGBM Model

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, roc_auc_score
from scipy.stats import spearmanr
from lightgbm import LGBMRegressor      # pip install lightgbm

data = pd.read_parquet('../data/processed/tracks_enriched.parquet')
orig_data= pd.read_parquet('../data/processed/orig_data.parquet')
target = 'popularity'

# ---- split ONCE; both stages use the same test set so it's never touched in training
train, test = train_test_split(data, test_size=0.2, random_state=42)

In [2]:
# ===== RAW popularity model =====
from sklearn.model_selection import GridSearchCV

target = 'popularity'
audio = ['energy','danceability','loudness','speechiness','instrumentalness',
         'acousticness','valence','tempo','liveness','duration_ms']   # <-- your actual audio cols
feats = audio + ['artist_fame_loo', 'track_genre']   # fame INCLUDED — see note below

X_tr, X_te = train[feats].copy(), test[feats].copy()
X_tr['track_genre'] = X_tr['track_genre'].astype('category')
X_te['track_genre'] = X_te['track_genre'].astype('category')
y_tr, y_te = train[target], test[target]

param_grid = {'n_estimators':[100,200,300], 'learning_rate': list(np.arange(0.02,0.13,0.03))}
gs = GridSearchCV(LGBMRegressor(verbose=-1, objective='tweedie'), param_grid, scoring='r2', cv=10, n_jobs=-1)
gs.fit(X_tr, y_tr)
print("best params:", gs.best_params_)

model = LGBMRegressor(**gs.best_params_, random_state=667, verbose=-1)
model.fit(X_tr, y_tr)
pred = model.predict(X_te)

print("R²      :", r2_score(y_te, pred))
print("MAE     :", mean_absolute_error(y_te, pred))
print("Spearman:", spearmanr(y_te, pred).correlation)

print(pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False))  # the payoff

best params: {'learning_rate': np.float64(0.11000000000000001), 'n_estimators': 100}
R²      : 0.7987467761013807
MAE     : 5.370640471132156
Spearman: 0.892940777160492
track_genre         823
artist_fame_loo     626
duration_ms         187
tempo               171
loudness            167
danceability        162
acousticness        161
speechiness         154
valence             150
instrumentalness    137
energy              132
liveness            130
dtype: int32


In [3]:
from sklearn.metrics import root_mean_squared_error
print("ctx RMSE :", root_mean_squared_error(y_te, pred))

ctx RMSE : 7.929047509609632
